## Classification Dataset

In [10]:
import pandas as pd

data_file_path = "sms_spam_collection/SMSSpamCollection.tsv"
df = pd.read_csv(data_file_path,  sep="\t", header=None, names=['Label', 'Text'])

spam_df = df[df['Label'] == 'spam']
truth_df = df[df['Label'] == 'ham'].sample(len(spam_df))
data_df = pd.concat([spam_df, truth_df])
data_df["Label"] = data_df["Label"].map({"ham": 0, "spam": 1})   

split = int(len(data_df) * 0.7)
balanced_df = data_df.sample(frac=1).reset_index(drop=True)
train_df = balanced_df[:split]
test_df = balanced_df[split:]
train_df.head(10)

,Label,Text
0,0,Friendship poem: Dear O Dear U R Not Near But ...
1,1,Someone U know has asked our dating service 2 ...
2,1,8007 25p 4 Alfie Moon's Children in Need song ...
3,0,Change again... It's e one next to escalator...
4,1,In The Simpsons Movie released in July 2007 na...
5,0,Ü say until like dat i dun buy ericsson oso ca...
6,0,Why you keeping me away like this
7,1,1000's of girls many local 2 u who r virgins 2...
8,0,Sorry completely forgot * will pop em round th...
9,0,"Hiya, had a good day? Have you spoken to since..."


In [11]:
import torch
from torch.utils.data import Dataset


def pad_to_max(tokens, max_len, pad_token_id):
    if len(tokens) < max_len:
        return tokens + [pad_token_id] * (max_len - len(tokens))
    return tokens[:max_len]


class SpamEmails(Dataset):
    def __init__(self, df, tokenizer, pad_token_id):
        super().__init__()

        self.data = df.copy()

        # tokenize the Text column
        self.data["Token"] = self.data["Text"].apply(tokenizer.encode)

        # compute lengths
        lengths = self.data["Token"].apply(len).sort_values().to_list()
        max_len = lengths[int(0.95 * len(lengths))]

        # pad/truncate
        self.data["Token"] = self.data["Token"].apply(
            lambda tokens: pad_to_max(tokens, max_len, pad_token_id)
        )

    def __getitem__(self, index):
        tokens = self.data.iloc[index]["Token"]
        label = self.data.iloc[index]["Label"]   # not "Labels"
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.data)

## Classifcation Model

In [12]:
import torch
import torch.nn as nn
from previous_chapters import GPTModel

weights_path = "../5 Pre-training/gpt2-small-124M.pth"

config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
}

model = GPTModel(config)
model.load_state_dict(torch.load(weights_path, weights_only=True))
model.eval()

for param in model.parameters():
    param.requires_grad = False

num_classes = 2
model.out_head = nn.Linear(config["emb_dim"], num_classes)

for param in model.out_head.parameters():
    param.requires_grad = True

for block in model.trf_blocks[-1:]:
    for param in block.parameters():
        param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

## Training

In [14]:
import tiktoken
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(42)

tokenizer = tiktoken.get_encoding('gpt2')
pad_token_id = 50256
train_dataset = SpamEmails(train_df, tokenizer, pad_token_id)
test_dataset = SpamEmails(test_df, tokenizer, pad_token_id)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=True,
)

In [15]:
epochs = 3
batch_size = 8
learning_rate = 3e-4
weight_decay = 0.01
eval_interval = 2

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
loss_fn = torch.nn.CrossEntropyLoss()
epoch_list = []
train_losses = []
test_losses = []

In [ ]:
for epoch in range(epochs):

    train_loss = 0
    model.train()

    for tokens, classification in train_loader:

        optimizer.zero_grad()

        preds = model(tokens)[:, -1, :]
        loss = loss_fn(preds, classification)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()


    if epoch % eval_interval == 0 or epoch == epochs - 1:

        test_loss = 0
        model.eval()

        with torch.no_grad():
            for tokens, classification in test_loader:

                preds = model(tokens)[:, -1, :]
                loss = loss_fn(preds, classification)
                test_loss += loss.item()

        train_loss /= len(train_loader)
        test_loss /= len(test_loader)

        epoch_list.append(epoch)
        train_losses.append(train_loss)
        test_losses.append(test_loss)

        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, test_loss={test_loss:.4f}")

        break

Epoch 1: train_loss=0.2802, test_loss=0.1429
